<a href="https://colab.research.google.com/github/Karna14314/Hugginface_models/blob/main/Fine_tune_Sentiment_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Colab!

```markdown
# Project: Fast DistilBERT Sentiment Fine-Tuning & Deployment
This notebook demonstrates a recruiter-ready NLP pipeline: fine-tuning a transformer model, adding explainability, and preparing for Hugging Face deployment.
```

In [ ]:
!pip install -q datasets transformers[torch] evaluate captum huggingface_hub

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

# 1. Load a subset of IMDb (5k training, 1k test for speed)
dataset = load_dataset("imdb")
train_sub = dataset["train"].shuffle(seed=42).select(range(5000))
test_sub = dataset["test"].shuffle(seed=42).select(range(1000))

# 2. Preprocessing
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_train = train_sub.map(tokenize_function, batched=True)
tokenized_test = test_sub.map(tokenize_function, batched=True)

# 3. Model & Metrics
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=2)
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
# 4. Fast Training (1 Epoch)
training_args = TrainingArguments(
    output_dir="sentiment-distilbert-fast",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    push_to_hub=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

trainer.train()

```markdown
### Explainability: Visualizing Word Importance
Using simple gradient-based saliency to show which words influenced the sentiment prediction.
```

In [ ]:
def get_explainability(text):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"]

    # Enable gradients for embeddings
    embeddings = model.get_input_embeddings()(input_ids)
    embeddings.retain_grad()

    outputs = model(inputs_embeds=embeddings, attention_mask=inputs["attention_mask"])
    score = torch.max(outputs.logits)
    score.backward()

    # Calculate saliency (norm of gradients)
    saliency = embeddings.grad.data.abs().sum(dim=-1).squeeze(0)
    saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min())

    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    return list(zip(tokens, saliency.tolist()))

sample_text = "This movie was absolutely fantastic and the acting was superb!"
explanation = get_explainability(sample_text)
print("Token Importance (Top 5):", sorted(explanation, key=lambda x: x[1], reverse=True)[:5])

```markdown
### Deploy to Hugging Face Spaces
To push to Spaces, you'll need your [Hugging Face Write Token](https://huggingface.co/settings/tokens).
```

In [ ]:
from huggingface_hub import notebook_login, HfApi

# Uncomment the next line to login
notebook_login()

# Code to push model
model.push_to_hub("ncncomplete/distilbert-imdb-sentiment")
tokenizer.push_to_hub("ncncomplete/distilbert-imdb-sentiment")